[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChrisW09/Python-for-AI-Driven-Automation/blob/main/10_agents_tools_mcp/38_designing_robust_tools.ipynb)

# 📓 Notebook 38 — Designing Robust Tools

> **Module:** Agents, Tools & MCP · **Estimated time:** 70–90 min · **Difficulty:** Intermediate → Advanced

An agent is only as good as its tools. Notebook 37 gave tools to a loop; here we make the **tools themselves** production-grade. A real tool is called by a model that may pass wrong types, missing fields, or hostile input — and may call it dozens of times. So a tool needs a **typed schema**, **input validation**, **structured errors**, **size limits**, an **approval gate** for dangerous actions, and the ability to run **in parallel**.

Everything runs offline. The patterns map 1:1 onto OpenAI/Anthropic function-calling and onto MCP tools (Notebook 39).

> 🎯 **Same copilot, one consequential new tool.** We're still building the **AI support copilot** from Notebook 37 — but its tools so far only *read* data, and reading is harmless. This notebook gives it `estimate_refund(order_id, pct)`, a tool that **moves money**. That one tool is our running example, and it's the perfect stress test: what should happen when the model sends `pct: "fifty"`, forgets the `order_id`, or — the one that keeps you up at night — asks for `pct: 9999`?

> 🧭 **Mental model for the whole notebook: the schema is the contract you advertise; the validator is the bouncer at the door.** A tool is *a function plus a schema the model reads* — and that schema is a promise the model can break. The model is a brilliant *suggester* of calls and a hopeless *guarantor* of them. So treat every argument it sends as **untrusted input** — exactly as you'd treat a value a stranger typed into a web form. The bouncer checks every guest against the list *before* they get through the door; nothing reaches your real `estimate_refund` code until it has.

## 🎯 Learning objectives

By the end you can:

1. Describe a tool with a **JSON-Schema** the model can read.
2. **Validate** arguments before executing — types, required fields, enums, ranges.
3. Return **structured results and errors** (never raw exceptions) in a consistent envelope.
4. Build a **ToolRegistry** that validates, dispatches, and logs every call.
5. Add a **human-in-the-loop approval gate** for sensitive tools.
6. Run independent tools **in parallel** and bound output size.

## ✅ Prerequisites

Notebook 37 (agent architectures), dictionaries & functions (NB 4–5), JSON (NB 4), tool schemas (NB 30).

## 1. Anatomy of a tool

A tool the model can use has four parts:

| Part | Why the model needs it |
|---|---|
| **name** | what to call |
| **description** | *when* to call it — the single most important field |
| **parameters** (JSON-Schema) | the shape of the arguments |
| **function** | your code that actually runs (the model never runs it) |

We bundle them in a small dataclass.

In [1]:
from dataclasses import dataclass, field
from typing import Callable, Any
import json, re, time

@dataclass
class Tool:
    name: str
    description: str
    parameters: dict          # JSON-Schema for the arguments object
    fn: Callable[..., Any]

    def schema(self) -> dict:
        """The provider-facing description — same idea everywhere, but field names
        differ by provider: `parameters` (OpenAI) / `input_schema` (Anthropic) /
        `inputSchema` (MCP)."""
        return {"name": self.name, "description": self.description,
                "parameters": self.parameters}

# Example tool: refund lookup over a tiny in-memory orders table
ORDERS = {"A-1": 19.0, "A-2": 49.0, "A-3": 5.0}

refund = Tool(
    name="estimate_refund",
    description="Estimate the refund owed for an order id. Use when a customer asks about money back.",
    parameters={
        "type": "object",
        "properties": {
            "order_id": {"type": "string", "description": "e.g. 'A-2'"},
            "pct":      {"type": "number", "minimum": 0, "maximum": 100,
                         "description": "percent to refund"},
        },
        "required": ["order_id", "pct"],
    },
    fn=lambda order_id, pct: {"order_id": order_id,
                              "refund": round(ORDERS[order_id] * pct / 100, 2)},
)
print(json.dumps(refund.schema(), indent=2))

{
  "name": "estimate_refund",
  "description": "Estimate the refund owed for an order id. Use when a customer asks about money back.",
  "parameters": {
    "type": "object",
    "properties": {
      "order_id": {
        "type": "string",
        "description": "e.g. 'A-2'"
      },
      "pct": {
        "type": "number",
        "minimum": 0,
        "maximum": 100,
        "description": "percent to refund"
      }
    },
    "required": [
      "order_id",
      "pct"
    ]
  }
}


## 2. Validate arguments *before* running

A model will, eventually, send `pct: "fifty"` or omit `order_id`. Running the function on bad input gives a confusing traceback. Instead, validate against the schema first and return a clear error the model can recover from.

We write a tiny validator (no extra dependency) covering the cases that actually bite: missing required fields, wrong types, enums, and numeric ranges.

In [2]:
_JSON_TYPES = {"string": str, "number": (int, float), "integer": int,
               "boolean": bool, "object": dict, "array": list}

def validate_args(schema: dict, args: dict) -> list[str]:
    """Return a list of human-readable problems ([] means valid)."""
    problems = []
    props = schema.get("properties", {})
    for req in schema.get("required", []):
        if req not in args:
            problems.append(f"missing required field '{req}'")
    for key, val in args.items():
        if key not in props:
            problems.append(f"unexpected field '{key}'"); continue
        spec = props[key]
        exp = spec.get("type")
        if exp and not isinstance(val, _JSON_TYPES.get(exp, object)):
            problems.append(f"'{key}' should be {exp}, got {type(val).__name__}"); continue
        if "enum" in spec and val not in spec["enum"]:
            problems.append(f"'{key}' must be one of {spec['enum']}")
        if isinstance(val, (int, float)):
            if "minimum" in spec and val < spec["minimum"]:
                problems.append(f"'{key}' below minimum {spec['minimum']}")
            if "maximum" in spec and val > spec["maximum"]:
                problems.append(f"'{key}' above maximum {spec['maximum']}")
    return problems

print("valid  :", validate_args(refund.parameters, {"order_id": "A-2", "pct": 50}))
print("bad    :", validate_args(refund.parameters, {"order_id": "A-2", "pct": 150}))
print("missing:", validate_args(refund.parameters, {"pct": 50}))
print("type   :", validate_args(refund.parameters, {"order_id": "A-2", "pct": "fifty"}))

valid  : []
bad    : ["'pct' above maximum 100"]
missing: ["missing required field 'order_id'"]
type   : ["'pct' should be number, got str"]


### 🔬 What actually happens — the model reads a SCHEMA and proposes a call

It is easy to assume the model *runs* your `estimate_refund` function. It does not. The model never sees your Python at all. What it sees is the **schema** — the JSON you printed in `refund.schema()`: a name, a description, and a typed list of parameters. From that text alone, the model writes back a **proposed call**: a tool name plus an arguments object. That proposal is just *data* — a little dict — that your code receives and decides what to do with.

```text
   YOUR SIDE                              MODEL SIDE
   ─────────                              ──────────
  Tool = function + SCHEMA
  ┌───────────────────────────┐
  │ name        estimate_refund│  ──advertise the SCHEMA──▶  the model READS it
  │ description "...refund..."  │      (name + description + typed params)
  │ params      order_id: str  │
  │             pct: number    │                                │
  │ fn          <your code>    │                                │  the model can't
  └───────────────────────────┘                                │  run fn — it can
                                                                 │  only emit text
                                                                 ▼
                                       proposed call (just data):
                                       {"name": "estimate_refund",
                                        "args": {"order_id": "A-2", "pct": 50}}
                                                │
        ◀───────── the model hands the proposal back to YOU ────┘
```

The model is a very good *suggester* of calls and a very bad *guarantor* of them. It read a description that said `pct` is a number from 0 to 100 — but nothing forces it to obey. It can send `pct: "fifty"` (wrong type), forget `order_id` (missing required field), or send `pct: 150` (out of range). So the schema is only half the contract. The other half is **you checking that the proposal actually honours it** before you act on it.

> 🧭 **The mental model, in the picture above.** This is the *"schema = contract, validator = bouncer"* idea from the intro made concrete. The **schema** is the contract you advertise (the left box); the **proposed call** is a guest showing up at the door (the right box); and the bouncer — which we build next — is what stands between that proposal and your real `fn`. Treat every tool argument the model sends as *untrusted input*, exactly as you would treat a value typed into a web form by a stranger.

### What a *good* schema looks like — and what validation must check

A good schema makes the model's job easy (it knows exactly when and how to call) and makes your job safe (you know exactly what to check). Four ingredients:

| Schema ingredient | Example | Why it matters |
|---|---|---|
| **clear name** | `estimate_refund` | The model matches the user's intent to this name. |
| **prescriptive description** | *"Use when a customer asks about money back."* | The single biggest driver of *whether* the model picks this tool. Say **when** to call it, not just what it does. |
| **typed parameters** | `order_id: string`, `pct: number` | Tells the model the shape — and tells your validator what types to enforce. |
| **which are required** | `required: ["order_id", "pct"]` | Distinguishes "must be present" from "optional with a default". |

Validation is the mirror image: for every ingredient the schema *advertises*, the validator *enforces* it on the incoming args. These are the four failure modes that actually bite in production:

| The model sends… | Schema rule it breaks | Validator's job |
|---|---|---|
| `{"pct": 50}` (no `order_id`) | `required` | reject: missing required field |
| `{"order_id": "A-2", "pct": "fifty"}` | type `number` | reject: wrong type |
| `{"order_id": "A-2", "pct": 150}` | `maximum: 100` | reject: out of range |
| `{"order_id": "A-2", "pct": 50, "x": 9}` | known `properties` only | reject: unexpected field |

The validator's *output* matters as much as its decision. A bad call should **never raise an uncaught exception** and should **never silently execute**. Instead it returns a clear, structured error — and that error gets handed *back to the model*, which can read it and correct its next attempt.

In [3]:
# 🧪 OFFLINE PROOF — self-contained. No API. Bad calls are reported, never raised.
# We define a tiny tool with a small SCHEMA, a validate() that checks
# required keys + types + ranges, and a guarded runner that decides
# accept-and-execute vs reject-with-error-back-to-the-model.

# 1) The SCHEMA we advertise to the model (this is the "contract").
schema_dd = {
    "type": "object",
    "properties": {
        "order_id": {"type": "string", "description": "e.g. 'A-2'"},
        "pct":      {"type": "number", "minimum": 0, "maximum": 100,
                     "description": "percent to refund"},
    },
    "required": ["order_id", "pct"],
}

# 2) The real action behind the tool (the model never runs this — we do).
ORDERS_DD = {"A-1": 19.0, "A-2": 49.0, "A-3": 5.0}
def estimate_refund_dd(order_id, pct):
    return {"order_id": order_id, "refund": round(ORDERS_DD[order_id] * pct / 100, 2)}

# 3) The bouncer: validate args against the schema. Returns [] if clean,
#    else a list of human-readable problems. No exceptions, ever.
_PYTYPES = {"string": str, "number": (int, float), "integer": int, "boolean": bool}
def validate(args, schema):
    problems = []
    props = schema.get("properties", {})
    for req in schema.get("required", []):              # required keys present?
        if req not in args:
            problems.append(f"missing required field '{req}'")
    for key, val in args.items():
        if key not in props:                            # only known fields allowed
            problems.append(f"unexpected field '{key}'"); continue
        spec = props[key]
        want = spec.get("type")
        if want and not isinstance(val, _PYTYPES.get(want, object)):  # right type?
            problems.append(f"'{key}' should be {want}, got {type(val).__name__}"); continue
        if isinstance(val, (int, float)):               # within range?
            if "minimum" in spec and val < spec["minimum"]:
                problems.append(f"'{key}' below minimum {spec['minimum']}")
            if "maximum" in spec and val > spec["maximum"]:
                problems.append(f"'{key}' above maximum {spec['maximum']}")
    return problems

# 4) The guard at the door: validate FIRST; only then execute.
def guarded_call(args):
    problems = validate(args, schema_dd)
    if problems:
        # structured error fed BACK to the model so it can correct (is_error=True)
        return {"ok": False, "error": "; ".join(problems)}
    return {"ok": True, "result": estimate_refund_dd(**args)}   # safe to act now

# 5) Run ONE good call and SEVERAL bad ones the model might really send.
proposals = [
    ("GOOD    ", {"order_id": "A-2", "pct": 50}),     # valid
    ("MISSING ", {"pct": 50}),                        # forgot order_id
    ("WRONGTY ", {"order_id": "A-2", "pct": "fifty"}),# string where number expected
    ("RANGE   ", {"order_id": "A-2", "pct": 150}),    # above maximum
    ("EXTRA   ", {"order_id": "A-2", "pct": 50, "note": "x"}),  # unexpected field
]
for label, args in proposals:
    out = guarded_call(args)
    if out["ok"]:
        print(f"{label} ACCEPT -> executed: {out['result']}")
    else:
        print(f"{label} REJECT -> error back to model: {out['error']}")


GOOD     ACCEPT -> executed: {'order_id': 'A-2', 'refund': 24.5}
MISSING  REJECT -> error back to model: missing required field 'order_id'
WRONGTY  REJECT -> error back to model: 'pct' should be number, got str
RANGE    REJECT -> error back to model: 'pct' above maximum 100
EXTRA    REJECT -> error back to model: unexpected field 'note'


### Reading the proof — accept the good, reject-and-explain the rest

Exactly one of the five proposals executed. The other four were stopped *at the door* and turned into a readable error string — no traceback, no half-finished refund, no `KeyError` leaking out. That is the whole discipline in four lines: **validate first, execute only on a clean bill of health.**

The rejected cases never touched `estimate_refund_dd`, so the real action (which here is harmless arithmetic, but in production is a DB write, a shell command, or a payment) was never reached with bad input. And crucially, each rejection produced a *message the model can act on*:

```text
  model proposes  ──▶  validate(args, schema)  ──▶  clean?  ──▶  EXECUTE  ──▶  result
                                                      │
                                                      └── problems ──▶  ERROR back to model
                                                                         "'pct' above maximum 100"
                                                                              │
                                                                              ▼
                                                               model reads it, retries with pct: 100
```

When you wire this to a real Claude tool call, the accept path returns a normal `tool_result`, and the reject path returns a `tool_result` with `is_error: true` carrying that same string — the model sees its mistake and corrects on the next turn instead of the whole agent crashing.

> 🎯 **The takeaway to keep.** A tool is **a function plus a schema the model reads** — and the schema is a *promise the model can break*. Your validator is the part that makes the promise real: it treats every argument as untrusted, accepts only what matches the contract, and hands a precise error back for everything else. Never pass unchecked model output into a real action.

> ⚠️ **Don't skip validation because "the model is usually right."** *Usually* is not *always*, and the one time it sends `pct: 9999` is the time it issues a 99× refund. Validation is cheap; an unguarded state-changing tool is a production incident waiting for its trigger.

---

### ✋ Quick exercise (~2 min) — Harden a schema with an `enum`

> 🧑‍🏫 *Live checkpoint — pause and try this before peeking at the solution.*

Your support copilot is getting a `set_priority(ticket_id, level)` tool. Write its arguments JSON-Schema so `level` is restricted to `["low", "high"]` and **both** fields are required, then use `validate_args` to confirm it rejects `level: "urgent"`.

```python
priority_schema = {
    "type": "object",
    "properties": { ... },   # ticket_id: string, level: string enum
    "required": [ ... ],
}
```

In [4]:
# ✍️ Your turn 👇
priority_schema = {
    "type": "object",
    "properties": {
        # add: ticket_id (string) and level (string, restricted to "low"/"high")
    },
    "required": [],          # which fields must always be present?
}
# print(validate_args(priority_schema, {"ticket_id": "T-9", "level": "urgent"}))

<details>
<summary>✅ <b>Solution</b></summary>

```python
priority_schema = {
    "type": "object",
    "properties": {
        "ticket_id": {"type": "string"},
        "level": {"type": "string", "enum": ["low", "high"]},
    },
    "required": ["ticket_id", "level"],
}
print("ok :", validate_args(priority_schema, {"ticket_id": "T-9", "level": "high"}))
print("bad:", validate_args(priority_schema, {"ticket_id": "T-9", "level": "urgent"}))
```

Putting the legal values in an `enum` makes them part of the contract, so `validate_args` rejects `"urgent"` with a readable message before the tool ever runs.
</details>

## 3. A consistent result envelope

Return the *same shape* whether a tool succeeds or fails. Models (and your logs) handle one predictable structure far better than a mix of values and exceptions:

```json
{"ok": true,  "result": {...}}
{"ok": false, "error": "estimate_refund: 'pct' above maximum 100"}
```

In [5]:
def ok(result):  return {"ok": True,  "result": result}
def err(message): return {"ok": False, "error": message}

def safe_call(tool: Tool, args: dict) -> dict:
    problems = validate_args(tool.parameters, args)
    if problems:
        return err(f"{tool.name}: " + "; ".join(problems))
    try:
        return ok(tool.fn(**args))
    except Exception as e:                      # never leak a raw traceback to the model
        return err(f"{tool.name} raised {type(e).__name__}: {e}")

print(safe_call(refund, {"order_id": "A-2", "pct": 50}))
print(safe_call(refund, {"order_id": "A-2", "pct": 150}))
print(safe_call(refund, {"order_id": "NOPE", "pct": 50}))   # KeyError -> structured error

{'ok': True, 'result': {'order_id': 'A-2', 'refund': 24.5}}
{'ok': False, 'error': "estimate_refund: 'pct' above maximum 100"}
{'ok': False, 'error': "estimate_refund raised KeyError: 'NOPE'"}


## 4. A ToolRegistry: validate, dispatch, log

In a real agent you have *many* tools. A registry gives you one place to register them, hand their schemas to the model, dispatch calls by name, and **log every invocation** for debugging and cost tracking.

In [6]:
class ToolRegistry:
    def __init__(self):
        self._tools: dict[str, Tool] = {}
        self.log: list[dict] = []

    def register(self, tool: Tool):
        self._tools[tool.name] = tool
        return self

    def schemas(self) -> list[dict]:
        return [t.schema() for t in self._tools.values()]

    def call(self, name: str, args: dict) -> dict:
        t0 = time.perf_counter()
        if name not in self._tools:
            out = err(f"no such tool '{name}' (have: {list(self._tools)})")
        else:
            out = safe_call(self._tools[name], args)
        self.log.append({"tool": name, "args": args, "ok": out["ok"],
                         "ms": round((time.perf_counter() - t0) * 1000, 2)})
        return out

reg = ToolRegistry().register(refund)
reg.register(Tool("list_orders", "List all order ids and amounts.",
                  {"type": "object", "properties": {}}, lambda: dict(ORDERS)))

print(reg.call("estimate_refund", {"order_id": "A-1", "pct": 100}))
print(reg.call("list_orders", {}))
print(reg.call("delete_db", {}))            # unknown tool -> structured error
print("\ncall log:", json.dumps(reg.log, indent=1))

{'ok': True, 'result': {'order_id': 'A-1', 'refund': 19.0}}
{'ok': True, 'result': {'A-1': 19.0, 'A-2': 49.0, 'A-3': 5.0}}
{'ok': False, 'error': "no such tool 'delete_db' (have: ['estimate_refund', 'list_orders'])"}

call log: [
 {
  "tool": "estimate_refund",
  "args": {
   "order_id": "A-1",
   "pct": 100
  },
  "ok": true,
  "ms": 0.0
 },
 {
  "tool": "list_orders",
  "args": {},
  "ok": true,
  "ms": 0.0
 },
 {
  "tool": "delete_db",
  "args": {},
  "ok": false,
  "ms": 0.0
 }
]


---

### ✋ Quick exercise (~2 min) — Register a read-only tool

> 🧑‍🏫 *Live checkpoint — pause and try this before peeking at the solution.*

Give the copilot a harmless `count_orders` tool that returns how many orders exist. Build a fresh `ToolRegistry`, `register` the tool (empty `properties` schema), call it **through the registry**, and confirm you get back the standard `ok` envelope.

In [7]:
# ✍️ Your turn 👇
my_reg = ToolRegistry()
# register a Tool named "count_orders" (no params) whose fn returns len(ORDERS),
# then call it through my_reg and print the envelope

<details>
<summary>✅ <b>Solution</b></summary>

```python
my_reg = ToolRegistry().register(
    Tool("count_orders", "Count how many orders currently exist.",
         {"type": "object", "properties": {}}, lambda: len(ORDERS)))
print(my_reg.call("count_orders", {}))
```

The registry validates the (empty) schema, dispatches by name, wraps the return value in the standard `{"ok": True, "result": ...}` envelope, and logs the call — all for free.
</details>

## 5. Tool selection among many

With one or two tools the model picks easily. With twenty, *descriptions* do the work — the model matches the user's intent to the best `description`. Offline, we mimic that with a keyword router; the lesson is the same: **a tool is only discoverable if its description says when to use it.**

In [8]:
STOP = {"the", "a", "an", "me", "all", "for", "is", "please", "show", "to", "of", "id", "ids"}

def route(user_msg: str, registry: ToolRegistry) -> str | None:
    """Stand-in for the model choosing a tool by reading its description.

    Real models do this semantically; here we match content words (ignoring
    stopwords) and count light stem overlaps so 'orders' matches 'order'.
    """
    msg = {w for w in re.findall(r"[a-z]+", user_msg.lower()) if w not in STOP}
    best, best_score = None, 0
    for schema in registry.schemas():
        desc = {w for w in re.findall(r"[a-z]+", schema["description"].lower()) if w not in STOP}
        score = sum(any(m == d or (len(m) >= 4 and (m.startswith(d) or d.startswith(m)))
                        for d in desc) for m in msg)
        if score > best_score:
            best, best_score = schema["name"], score
    return best

print("‘how much money back for A-2?’ ->", route("how much money back for A-2", reg))
print("‘list all the orders’         ->", route("list all the orders for me", reg))

‘how much money back for A-2?’ -> estimate_refund
‘list all the orders’         -> list_orders


## 6. Approval gate: human-in-the-loop for dangerous tools

Reading data is safe; **issuing a refund, sending an email, or deleting a row is not.** Mark sensitive tools and require explicit approval before they execute. The gate is a callback — in production it's a Slack button or a UI prompt; here it's a simple policy function.

In [9]:
SENSITIVE = {"estimate_refund"}     # tools that change money/state

def gated_call(registry, name, args, approver):
    if name in SENSITIVE:
        if not approver(name, args):
            return err(f"'{name}' denied by approver")
    return registry.call(name, args)

def auto_approver(name, args):
    # Example policy: auto-approve small refunds, escalate large ones.
    decision = not (name == "estimate_refund" and args.get("pct", 0) > 50)
    print(f"   approver: {name}({args}) -> {'APPROVE' if decision else 'DENY'}")
    return decision

print(gated_call(reg, "estimate_refund", {"order_id": "A-1", "pct": 25}, auto_approver))
print(gated_call(reg, "estimate_refund", {"order_id": "A-1", "pct": 90}, auto_approver))

   approver: estimate_refund({'order_id': 'A-1', 'pct': 25}) -> APPROVE
{'ok': True, 'result': {'order_id': 'A-1', 'refund': 4.75}}
   approver: estimate_refund({'order_id': 'A-1', 'pct': 90}) -> DENY
{'ok': False, 'error': "'estimate_refund' denied by approver"}


---

### ✋ Quick exercise (~2 min) — Tighten the approval policy

> 🧑‍🏫 *Live checkpoint — pause and try this before peeking at the solution.*

Compliance now wants a human to sign off on any refund above 25%. Write a `strict_approver(name, args)` that **denies** `estimate_refund` when `pct > 25` and approves otherwise, then use `gated_call` to show a 40% refund is denied but a 10% one passes.

In [10]:
# ✍️ Your turn 👇
def strict_approver(name, args):
    # deny estimate_refund when pct > 25, otherwise approve
    ...

# print(gated_call(reg, "estimate_refund", {"order_id": "A-1", "pct": 40}, strict_approver))
# print(gated_call(reg, "estimate_refund", {"order_id": "A-1", "pct": 10}, strict_approver))

<details>
<summary>✅ <b>Solution</b></summary>

```python
def strict_approver(name, args):
    return not (name == "estimate_refund" and args.get("pct", 0) > 25)

print(gated_call(reg, "estimate_refund", {"order_id": "A-1", "pct": 40}, strict_approver))
print(gated_call(reg, "estimate_refund", {"order_id": "A-1", "pct": 10}, strict_approver))
```

`gated_call` consults the approver *before* running a `SENSITIVE` tool; returning `False` produces a structured "denied" error instead of moving money, so the 40% refund is blocked while the 10% one goes through.
</details>

## 7. Parallel tool calls

When a model requests several *independent* tools (e.g. look up three orders at once), run them concurrently instead of in series. Bound the pool so you never launch unbounded work.

In [11]:
from concurrent.futures import ThreadPoolExecutor

def call_many(registry, calls: list[tuple[str, dict]], max_workers: int = 4) -> list[dict]:
    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        futures = [pool.submit(registry.call, name, args) for name, args in calls]
        return [f.result() for f in futures]

batch = [("estimate_refund", {"order_id": oid, "pct": 100}) for oid in ORDERS]
for r in call_many(reg, batch):
    print(r)

{'ok': True, 'result': {'order_id': 'A-1', 'refund': 19.0}}
{'ok': True, 'result': {'order_id': 'A-2', 'refund': 49.0}}
{'ok': True, 'result': {'order_id': 'A-3', 'refund': 5.0}}


## 8. Bound the output

A tool that returns a 50,000-row table will blow your context window and your bill. Cap tool output — truncate, paginate, or summarise — and tell the model you did.

In [12]:
def bounded(result, max_items: int = 3) -> dict:
    if isinstance(result, dict) and len(result) > max_items:
        keep = dict(list(result.items())[:max_items])
        return {"truncated": True, "shown": max_items, "total": len(result), "items": keep}
    return {"truncated": False, "items": result}

big = {f"row-{i}": i for i in range(100)}
print(bounded(big))

{'truncated': True, 'shown': 3, 'total': 100, 'items': {'row-0': 0, 'row-1': 1, 'row-2': 2}}


---

### ✋ Quick exercise (~2 min) — Cap a noisy tool result

> 🧑‍🏫 *Live checkpoint — pause and try this before peeking at the solution.*

A catalog lookup could return hundreds of rows and blow your context window. Use `bounded` to cap a 200-row result to the first **5** items, and confirm the envelope reports `truncated: True` with the correct `total`.

```python
catalog = {f"SKU-{i}": i for i in range(200)}
```

In [13]:
# ✍️ Your turn 👇
catalog = {f"SKU-{i}": i for i in range(200)}
# use bounded(...) to keep only the first 5 items, then print the envelope

<details>
<summary>✅ <b>Solution</b></summary>

```python
catalog = {f"SKU-{i}": i for i in range(200)}
print(bounded(catalog, max_items=5))
```

`bounded` keeps only the first `max_items` entries and reports `truncated: True` along with `shown` and `total`, so the model knows the output was capped (and could ask for more) instead of silently losing 195 rows.
</details>

## 🧪 Practice exercises

### Exercise 1 — ⭐ Add an `enum` field

Add a `currency` argument to a tool, restricted to `["EUR", "USD"]`, and show `validate_args` rejects `"GBP"`.

<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
pay_schema = {"type": "object",
              "properties": {"currency": {"type": "string", "enum": ["EUR", "USD"]}},
              "required": ["currency"]}
print("USD:", validate_args(pay_schema, {"currency": "USD"}))
print("GBP:", validate_args(pay_schema, {"currency": "GBP"}))
```

**Reasoning:** An `enum` makes the legal values part of the *contract*, so `validate_args` can reject `"GBP"` with a readable message before the tool ever runs — the model gets feedback it can act on instead of a stack trace.
</details>

### Exercise 2 — ⭐⭐ Count failures in the log

Write `failure_rate(registry)` returning the fraction of logged calls where `ok` is False.

<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
def failure_rate(registry) -> float:
    if not registry.log: return 0.0
    fails = sum(1 for e in registry.log if not e["ok"])
    return round(fails / len(registry.log), 3)

print("failure rate so far:", failure_rate(reg))
```

**Reasoning:** The registry already logs an `ok` flag for every call, so the failure rate is a single pass over `registry.log`. Guarding the empty-log case avoids a division by zero on a fresh registry.
</details>

### Exercise 3 — ⭐⭐ A read-only registry view

Return just the *names and descriptions* of registered tools — what you'd show a user as "what can this agent do?".

<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
def capabilities(registry) -> list[str]:
    return [f"{s['name']}: {s['description']}" for s in registry.schemas()]
print("\n".join(capabilities(reg)))
```

**Reasoning:** `registry.schemas()` already exposes exactly the public surface (names + descriptions), so the capability list is a read-only projection — nothing internal leaks, and it doubles as the "what can this agent do?" text a UI would show.
</details>

### Exercise 4 — ⭐⭐ Debug me 🐞

This call is supposed to flag the too-large `pct` — but instead of reporting the problem, the validator *crashes* with `AttributeError: 'str' object has no attribute 'items'`. The args arrived as a JSON *string*, not a dict. Find and fix it (expand the solution below when ready).

In [14]:
# 🐞 BUG (INTENTIONALLY ERRORS): args passed as a JSON *string*, not a dict.
bad = '{"order_id": "A-2", "pct": 999}'
problems = validate_args(refund.parameters, bad)   # AttributeError: str has no .items / wrong behaviour
print("problems:", problems)
assert problems, "expected the out-of-range pct to be caught"

AttributeError: 'str' object has no attribute 'items'

<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
# ✅ Fix: parse the JSON to a dict first — tools receive dicts, not strings.
bad = json.loads('{"order_id": "A-2", "pct": 999}')
print("problems:", validate_args(refund.parameters, bad))
```

**Reasoning:** `json.loads` turns the wire-format string into the dict the validator expects; only then can `validate_args` walk `args.items()` and catch the out-of-range `pct`. Rule of thumb: parse at the boundary — tools (and validators) receive dicts, never raw JSON strings.
</details>

## 🧠 Stretch exercises

### Stretch A — ⭐⭐⭐ Idempotency keys

Make `estimate_refund` idempotent: a repeated call with the same `idempotency_key` returns the cached result instead of recomputing (so a retried request never double-refunds).

<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
_seen: dict[str, dict] = {}
def idempotent_refund(order_id, pct, idempotency_key):
    if idempotency_key in _seen:
        return {**_seen[idempotency_key], "replayed": True}
    res = {"order_id": order_id, "refund": round(ORDERS[order_id] * pct / 100, 2)}
    _seen[idempotency_key] = res
    return res

print(idempotent_refund("A-2", 50, "req-1"))
print(idempotent_refund("A-2", 50, "req-1"))   # replayed, not recomputed
```

**Reasoning:** The cache is keyed on the caller-supplied `idempotency_key`, so a retried request replays the stored result instead of recomputing it — a retry can never refund twice. The `replayed` flag makes the replay visible for debugging.
</details>

### Stretch B — ⭐⭐⭐ Retry with backoff

Wrap a flaky tool so it retries on failure with exponential backoff, up to N attempts.

<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
def with_retry(fn, attempts=3, base=0.001):
    def wrapped(*a, **k):
        for i in range(attempts):
            out = fn(*a, **k)
            if out.get("ok", True):
                return out
            time.sleep(base * (2 ** i))
        return out
    return wrapped

calls = {"n": 0}
def flaky():
    calls["n"] += 1
    return ok("done") if calls["n"] >= 2 else err("transient")

print(with_retry(flaky)())          # succeeds on the 2nd try
print("attempts made:", calls["n"])
```

**Reasoning:** The wrapper retries only while the envelope reports failure, sleeping `base * 2**i` between attempts (exponential backoff). After the last attempt it returns the failing envelope rather than raising — consistent with the never-leak-exceptions rule.
</details>

### Stretch C — ⭐⭐⭐ A timeout wrapper

Stop waiting for a tool call that runs longer than `seconds` and return a structured timeout error. One caveat to design around: Python threads **can't be force-killed**, so the wrapper *abandons* the overrun work — the thread may keep running in the background; we just stop waiting for its result.

<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
from concurrent.futures import ThreadPoolExecutor, TimeoutError as FuturesTimeout

def call_with_timeout(fn, seconds=0.05, *args, **kwargs):
    # Create the executor OUTSIDE a `with` block: the context manager's exit
    # calls shutdown(wait=True), which would block until the worker finishes —
    # defeating the timeout entirely.
    pool = ThreadPoolExecutor(max_workers=1)
    fut = pool.submit(fn, *args, **kwargs)
    try:
        result = fut.result(timeout=seconds)
    except FuturesTimeout:
        # Threads can't be killed — ABANDON the work: stop waiting, let nothing
        # else start, and return immediately.
        pool.shutdown(wait=False, cancel_futures=True)
        return err(f"timeout after {seconds}s")
    except Exception as e:               # a real bug is NOT a timeout
        pool.shutdown(wait=False)
        return err(f"{type(e).__name__}: {e}")
    pool.shutdown(wait=False)
    return ok(result)

t0 = time.perf_counter()
print(call_with_timeout(lambda: sum(range(1000)), 0.05))   # fast  -> ok
print(call_with_timeout(lambda: time.sleep(1), 0.02))      # slow  -> timeout
print(f"elapsed: {time.perf_counter() - t0:.2f}s — we did NOT wait the full 1s")
print(call_with_timeout(lambda: 1 / 0, 0.05))              # crash -> reported as itself
```

**Reasoning:** The executor is created *outside* `with` because the context manager's exit blocks until the worker finishes — a 1 s sleep with a 0.02 s timeout would still take ~1 s. After a `TimeoutError` we call `shutdown(wait=False, cancel_futures=True)` and return at once; the overrun thread is *abandoned*, not killed (Python can't force-kill threads). Catching `TimeoutError` separately also keeps real bugs (like `ZeroDivisionError`) labelled as what they are instead of fake timeouts.
</details>

### Stretch D — ⭐⭐⭐ Schema-driven argument coercion

Models sometimes send `"50"` where a number is wanted. Coerce string→number when the schema says `number`/`integer`, *then* validate.

#### ✅ Solution

This solution stays as a *runnable* code cell (not collapsed) because the bonus mini-project below reuses `coerce`.

In [15]:
def coerce(schema, args):
    out = dict(args)
    for k, spec in schema.get("properties", {}).items():
        if k in out and spec.get("type") in ("number", "integer") and isinstance(out[k], str):
            try: out[k] = float(out[k]) if spec["type"] == "number" else int(out[k])
            except ValueError: pass
    return out

raw = {"order_id": "A-2", "pct": "50"}
fixed = coerce(refund.parameters, raw)
print("coerced:", fixed, "| valid:", validate_args(refund.parameters, fixed))

coerced: {'order_id': 'A-2', 'pct': 50.0} | valid: []


## 🎁 Bonus mini-project — a hardened tool call

Combine everything into one `production_call(registry, name, args, approver)` that **coerces → validates → gates → dispatches → logs**, and returns the standard envelope. This is the function an agent loop should actually call.

<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
def production_call(registry, name, args, approver=lambda n, a: True):
    if name not in registry._tools:
        return err(f"no such tool '{name}'")
    schema = registry._tools[name].parameters
    args = coerce(schema, args)
    problems = validate_args(schema, args)
    if problems:
        return err(f"{name}: " + "; ".join(problems))
    if name in SENSITIVE and not approver(name, args):
        return err(f"'{name}' denied")
    return registry.call(name, args)

print(production_call(reg, "estimate_refund", {"order_id": "A-3", "pct": "10"}, auto_approver))
print(production_call(reg, "estimate_refund", {"order_id": "A-3", "pct": 200}, auto_approver))
```

**Reasoning:** Ordering is the whole point: coerce *before* validating (so `"50"` becomes `50.0` and passes the range check), validate *before* gating (no point asking a human to approve garbage), and gate *before* dispatching. Every failure exits through the same `err(...)` envelope, so the caller — an agent loop — only ever sees one shape.
</details>

## 🧠 Key takeaways

> 🧭 **Back to the bouncer.** Trace `estimate_refund` through this whole notebook and you'll see one idea, layered. The **schema** advertised the contract; the **validator** was the bouncer that rejected `pct: "fifty"` and `pct: 9999` at the door; the **envelope** made every accept *and* reject look the same to the caller; the **registry** logged every guest who came through; the **approval gate** added a human bouncer for the calls that move real money; and **coercion, idempotency, timeouts and bounded output** handled the awkward guests who *almost* belong. Same tool, same door, ever-tighter security. The lesson that outlives this notebook: **never pass unchecked model output into a real action.** Our copilot can now *act* safely — next we standardise how the model even *reaches* tools like this.

1. A tool = **name + description + JSON-Schema + function**; the *description* is what makes it discoverable.
2. **Validate arguments before executing** — missing fields, wrong types, enums, ranges.
3. Always return a **consistent envelope** (`{ok, result|error}`); never leak a raw traceback to the model.
4. A **ToolRegistry** centralises dispatch and logs every call for debugging and cost control.
5. Gate **sensitive** tools behind human (or policy) **approval**.
6. Run independent calls **in parallel**, **bound** output size, and make state-changing tools **idempotent**.
7. This exact tool shape is what MCP standardises — Notebook 39.

## ✅ Self-assessment

- [ ] Write a JSON-Schema for a tool's arguments
- [ ] Validate args and produce readable error messages
- [ ] Wrap a tool so it returns a consistent ok/error envelope
- [ ] Build a registry that dispatches by name and logs calls
- [ ] Add an approval gate for a sensitive tool
- [ ] Run several tools in parallel and bound their output

## 🚀 Next step

Continue with **Notebook 39 — The Model Context Protocol (MCP)**, which standardises *exactly* these tools (plus resources and prompts) so any host — Claude Desktop, Claude Code, your own app — can use them over a common wire protocol.